# Previsão da Taxa de Câmbio USD/EUR com Aprendizado de Máquina
### TCC — MBA em Inteligência Artificial e Big Data

**Objetivo:** Comparar modelos preditivos (ARIMA, Random Forest, LSTM) para prever a variação diária (t+1) e semanal (t+5) do par USD/EUR, incorporando variáveis de eventos geopolíticos como features adicionais.

**Estrutura do notebook:**
1. Instalação e importação de bibliotecas
2. Coleta e visualização dos dados
3. Criação da variável de eventos geopolíticos
4. Análise exploratória (EDA)
5. Pré-processamento
6. Modelo ARIMA (baseline)
7. Modelo Random Forest
8. Modelo LSTM
9. Comparação de resultados — t+1 e t+5
10. Conclusões

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalar bibliotecas não disponíveis por padrão no Kaggle
!pip install yfinance pmdarima -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import yfinance as yf

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import pmdarima as pm  # auto_arima

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import math

# Reprodutibilidade
np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
print('Bibliotecas carregadas com sucesso.')

## 2. Coleta e Visualização dos Dados

In [ ]:
# Baixar dados diários do par EUR/USD via Yahoo Finance
# Período: jan/2010 a mai/2025 — captura crise europeia, Brexit, COVID, Ucrânia, Israel/Irã
ticker = yf.download('EURUSD=X', start='2010-01-01', end='2025-05-01', auto_adjust=True)

# Usar apenas o preço de fechamento
df = ticker[['Close']].copy()
df.columns = ['EURUSD']
df.dropna(inplace=True)

print(f'Período: {df.index.min().date()} a {df.index.max().date()}')
print(f'Total de observações: {len(df)}')
df.tail()

In [ ]:
# Visualização da série completa
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['EURUSD'], color='#6A4C93', linewidth=0.8)
ax.set_title('Taxa de Câmbio EUR/USD — 2010 a 2025', fontsize=14)
ax.set_ylabel('EUR/USD')
ax.set_xlabel('Data')
plt.tight_layout()
plt.savefig('/kaggle/working/serie_eurusd.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

## 3. Variável de Eventos Geopolíticos

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# TABELA DE EVENTOS GEOPOLÍTICOS
# Critério de seleção: eventos com impacto documentado em mercados cambiais.
# Janela de impacto: ±5 dias úteis em torno do evento (10 dias no total).
# Referência para as datas: ajuste conforme fontes primárias utilizadas no TCC.
# ──────────────────────────────────────────────────────────────────────────────

events = [
    # (data_central, nome_do_evento)
    ('2010-05-02', 'Crise dívida grega — 1º pacote resgate'),
    ('2011-08-05', 'Rebaixamento EUA pela S&P'),
    ('2014-03-18', 'Anexação da Crimeia pela Rússia'),
    ('2016-06-24', 'Brexit — resultado do referendo'),
    ('2020-03-11', 'COVID-19 declarado pandemia pela OMS'),
    ('2022-02-24', 'Invasão russa da Ucrânia'),
    ('2023-10-07', 'Ataque Hamas a Israel — início do conflito em Gaza'),
    ('2024-04-14', 'Ataque direto do Irã a Israel'),
    ('2025-04-01', 'Escalada tensão EUA-Irã — tarifas e sanções'),  # ajuste a data exata
]

# Criar coluna de evento (0 = sem evento, 1 = janela de evento)
df['evento'] = 0
JANELA = 5  # dias úteis para cada lado

for date_str, name in events:
    center = pd.Timestamp(date_str)
    mask = (df.index >= center - pd.Timedelta(days=JANELA)) & \
           (df.index <= center + pd.Timedelta(days=JANELA))
    df.loc[mask, 'evento'] = 1
    print(f'{name}: {mask.sum()} dias marcados')

print(f'\nTotal de dias com evento: {df["evento"].sum()} ({df["evento"].mean()*100:.1f}% da série)')

In [ ]:
# Visualizar a série com eventos marcados
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['EURUSD'], color='#6A4C93', linewidth=0.8, label='EUR/USD')

# Sombrear regiões de evento
in_event = False
start = None
for i, (idx, row) in enumerate(df.iterrows()):
    if row['evento'] == 1 and not in_event:
        start = idx
        in_event = True
    elif row['evento'] == 0 and in_event:
        ax.axvspan(start, idx, alpha=0.25, color='tomato')
        in_event = False
if in_event:
    ax.axvspan(start, df.index[-1], alpha=0.25, color='tomato')

# Linha vertical nos eventos principais
for date_str, name in events:
    ax.axvline(pd.Timestamp(date_str), color='red', linewidth=0.8, linestyle='--', alpha=0.6)

ax.set_title('EUR/USD com Janelas de Eventos Geopolíticos', fontsize=14)
ax.set_ylabel('EUR/USD')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/eventos_geopoliticos.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

## 4. Análise Exploratória (EDA)

In [ ]:
# Retornos logarítmicos diários
df['retorno'] = np.log(df['EURUSD'] / df['EURUSD'].shift(1))
df.dropna(inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].plot(df.index, df['retorno'], color='#6A4C93', linewidth=0.6)
axes[0].set_title('Retornos Logarítmicos Diários')
axes[1].hist(df['retorno'], bins=80, color='#6A4C93', edgecolor='white')
axes[1].set_title('Distribuição dos Retornos')
plt.tight_layout()
plt.savefig('/kaggle/working/retornos_distribuicao.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

print('Estatísticas descritivas dos retornos:')
print(df['retorno'].describe().round(6))

# Comparar volatilidade em dias de evento vs dias normais
vol_evento  = df[df['evento']==1]['retorno'].std()
vol_normal  = df[df['evento']==0]['retorno'].std()
print(f'\nVolatilidade em dias de evento: {vol_evento:.6f}')
print(f'Volatilidade em dias normais:  {vol_normal:.6f}')
print(f'Razão evento/normal: {vol_evento/vol_normal:.2f}x')

In [ ]:
# Teste de estacionariedade (ADF) na série de preços e nos retornos
def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f'--- ADF Test: {name} ---')
    print(f'  Estatística ADF : {result[0]:.4f}')
    print(f'  p-valor         : {result[1]:.4f}')
    print(f'  Conclusão       : {"Estacionária" if result[1] < 0.05 else "NÃO estacionária"}')
    print()

adf_test(df['EURUSD'], 'Preço EUR/USD')
adf_test(df['retorno'], 'Retorno logarítmico')

In [ ]:
# ACF e PACF dos retornos (com zoom no eixo Y)
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df['retorno'].dropna(), color='#6A4C93', vlines_kwargs={'colors': '#6A4C93', 'linewidth': 1}, lags=40, ax=axes[0], title='ACF — Retornos')
plot_pacf(df['retorno'].dropna(), color='#6A4C93', vlines_kwargs={'colors': '#6A4C93', 'linewidth': 1}, lags=40, ax=axes[1], title='PACF — Retornos')

# Zoom: limita o eixo Y para revelar os valores pequenos
for ax in axes:
    for line in ax.lines:
        line.set_color('#6A4C93')
    for collection in ax.collections:
        collection.set_facecolor('#6A4C93')
        collection.set_alpha(0.2)
    
    ax.set_ylim(-0.1, 0.1)

plt.tight_layout()
plt.savefig('/kaggle/working/acf_pacf.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

## 5. Pré-processamento e Feature Engineering

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Features técnicas + variável de evento
# Usadas pelos modelos Random Forest e LSTM
# ──────────────────────────────────────────────────────────────────────────────

df['ret_lag1'] = df['retorno'].shift(1)
df['ret_lag2'] = df['retorno'].shift(2)
df['ret_lag5'] = df['retorno'].shift(5)
df['media_mov5']  = df['EURUSD'].rolling(5).mean()
df['media_mov20'] = df['EURUSD'].rolling(20).mean()
df['vol_5d']      = df['retorno'].rolling(5).std()   # volatilidade realizada 5 dias
df['vol_20d']     = df['retorno'].rolling(20).std()

# Alvo t+1: retorno do dia seguinte
df['alvo_t1'] = df['retorno'].shift(-1)

# Alvo t+5: retorno acumulado dos próximos 5 dias úteis
df['alvo_t5'] = df['retorno'].rolling(5).sum().shift(-5)

df.dropna(inplace=True)

FEATURES = ['ret_lag1', 'ret_lag2', 'ret_lag5',
            'media_mov5', 'media_mov20',
            'vol_5d', 'vol_20d', 'evento']

print(f'Shape do dataframe final: {df.shape}')
df[FEATURES + ['alvo_t1', 'alvo_t5']].tail()

In [ ]:
# Divisão treino/teste: 80% treino, 20% teste (sem embaralhamento — série temporal)
SPLIT = int(len(df) * 0.80)

train = df.iloc[:SPLIT]
test  = df.iloc[SPLIT:]

print(f'Treino: {train.index.min().date()} a {train.index.max().date()} ({len(train)} obs)')
print(f'Teste : {test.index.min().date()}  a {test.index.max().date()}  ({len(test)} obs)')

## 6. Modelo ARIMA (Baseline)

In [ ]:
# auto_arima seleciona automaticamente a ordem (p, d, q)
# Usa critério AIC para minimização
print('Selecionando ordem ARIMA... (pode demorar ~1 min)')
arima_model = pm.auto_arima(
    train['retorno'],
    seasonal=False,
    information_criterion='aic',
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore'
)
print(arima_model.summary())

In [ ]:
# Previsão t+1 no conjunto de teste (rolling forecast)
history = list(train['retorno'])
preds_arima_t1 = []

for i in range(len(test)):
    model_fit = arima_model.fit(history)
    yhat = model_fit.predict(n_periods=1)[0]
    preds_arima_t1.append(yhat)
    history.append(test['retorno'].iloc[i])
    if i % 100 == 0:
        print(f'  {i}/{len(test)} previsões concluídas...')

preds_arima_t1 = np.array(preds_arima_t1)
real_t1 = test['alvo_t1'].values

# Previsão t+5: somar 5 previsões consecutivas de 1 passo
preds_arima_t5 = []
history2 = list(train['retorno'])
for i in range(len(test)):
    model_fit = arima_model.fit(history2)
    yhat5 = model_fit.predict(n_periods=5).sum()
    preds_arima_t5.append(yhat5)
    history2.append(test['retorno'].iloc[i])

preds_arima_t5 = np.array(preds_arima_t5)
real_t5 = test['alvo_t5'].values

print('ARIMA — previsões concluídas.')

## 7. Modelo Random Forest

In [ ]:
X_train = train[FEATURES]
X_test  = test[FEATURES]

# ── t+1 ──
rf_t1 = RandomForestRegressor(n_estimators=300, max_depth=8,
                               min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_t1.fit(X_train, train['alvo_t1'])
preds_rf_t1 = rf_t1.predict(X_test)

# ── t+5 ──
rf_t5 = RandomForestRegressor(n_estimators=300, max_depth=8,
                               min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_t5.fit(X_train, train['alvo_t5'])
preds_rf_t5 = rf_t5.predict(X_test)

# Importância das variáveis
feat_imp = pd.Series(rf_t1.feature_importances_, index=FEATURES).sort_values(ascending=False)
feat_imp.plot(kind='bar', color='#6A4C93', figsize=(10, 4),
              title='Importância das Variáveis — Random Forest (t+1)')
plt.tight_layout()
plt.savefig('/kaggle/working/importancia_rf.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

print('Random Forest — treinamento concluído.')

## 7.5 Modelo Elastic Net

In [ ]:
# ============================================================
# CÉLULA — Treinamento Elastic Net (com padronização)
# ============================================================

from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

X_train = train[FEATURES]
X_test  = test[FEATURES]

scaler_enet = StandardScaler()
X_train_scaled = scaler_enet.fit_transform(X_train)
X_test_scaled  = scaler_enet.transform(X_test)

l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

enet_t1 = ElasticNetCV(
    l1_ratio=l1_ratios, alphas=None, n_alphas=100, eps=1e-4,
    cv=tscv, max_iter=20000, random_state=42
)
enet_t1.fit(X_train_scaled, train['alvo_t1'])
preds_enet_t1 = enet_t1.predict(X_test_scaled)

print(f"Elastic Net t+1 — l1_ratio ótimo: {enet_t1.l1_ratio_:.3f}")
print(f"Elastic Net t+1 — alpha ótimo:    {enet_t1.alpha_:.6f}")

enet_t5 = ElasticNetCV(
    l1_ratio=l1_ratios, alphas=None, n_alphas=100, eps=1e-4,
    cv=tscv, max_iter=20000, random_state=42
)
enet_t5.fit(X_train_scaled, train['alvo_t5'])
preds_enet_t5 = enet_t5.predict(X_test_scaled)

print(f"\nElastic Net t+5 — l1_ratio ótimo: {enet_t5.l1_ratio_:.3f}")
print(f"Elastic Net t+5 — alpha ótimo:    {enet_t5.alpha_:.6f}")

coef_t1 = pd.Series(enet_t1.coef_, index=FEATURES).sort_values(key=abs, ascending=False)
print("\nCoeficientes Elastic Net (t+1, em escala padronizada):")
print(coef_t1.round(6))

print("\nElastic Net — treinamento concluído.")

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['tomato' if v < 0 else '#6A4C93' for v in coef_t1.values]
ax.bar(coef_t1.index, coef_t1.values, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes Elastic Net (t+1) — features selecionadas', fontsize=13)
ax.set_ylabel('Coeficiente')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('/kaggle/working/coeficientes_enet.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

## 8. Modelo LSTM

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Preparação para LSTM: janela deslizante de 20 dias
# A LSTM recebe sequências 3D: (amostras, janela_temporal, features)
# ──────────────────────────────────────────────────────────────────────────────

WINDOW = 20

scaler_lstm_features = MinMaxScaler()
scaled_features = scaler_lstm_features.fit_transform(df[FEATURES])

scaler_t1 = MinMaxScaler()
scaler_t5 = MinMaxScaler()
scaled_t1 = scaler_t1.fit_transform(df[['alvo_t1']])
scaled_t5 = scaler_t5.fit_transform(df[['alvo_t5']])

def create_sequences(X, y, window):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_seq, y_seq_t1 = create_sequences(scaled_features, scaled_t1.flatten(), WINDOW)
_,     y_seq_t5 = create_sequences(scaled_features, scaled_t5.flatten(), WINDOW)

split_seq = int(len(X_seq) * 0.80)
X_tr, X_te   = X_seq[:split_seq], X_seq[split_seq:]
y_tr_t1, y_te_t1 = y_seq_t1[:split_seq], y_seq_t1[split_seq:]
y_tr_t5, y_te_t5 = y_seq_t5[:split_seq], y_seq_t5[split_seq:]

print(f'Shape das sequências de treino: {X_tr.shape}')

In [ ]:
def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

es = EarlyStopping(patience=10, restore_best_weights=True)

# ── LSTM t+1 ──
print('Treinando LSTM t+1...')
lstm_t1 = build_lstm((WINDOW, len(FEATURES)))
hist_t1 = lstm_t1.fit(X_tr, y_tr_t1, epochs=100, batch_size=32,
                       validation_split=0.1, callbacks=[es], verbose=0)
preds_lstm_t1_sc = lstm_t1.predict(X_te).flatten()
preds_lstm_t1 = scaler_t1.inverse_transform(preds_lstm_t1_sc.reshape(-1,1)).flatten()

# ── LSTM t+5 ──
print('Treinando LSTM t+5...')
lstm_t5 = build_lstm((WINDOW, len(FEATURES)))
hist_t5 = lstm_t5.fit(X_tr, y_tr_t5, epochs=100, batch_size=32,
                       validation_split=0.1, callbacks=[es], verbose=0)
preds_lstm_t5_sc = lstm_t5.predict(X_te).flatten()
preds_lstm_t5 = scaler_t5.inverse_transform(preds_lstm_t5_sc.reshape(-1,1)).flatten()

# Alinhar tamanhos (LSTM perde WINDOW observações no início)
real_lstm_t1 = test['alvo_t1'].values[-len(preds_lstm_t1):]
real_lstm_t5 = test['alvo_t5'].values[-len(preds_lstm_t5):]

print('LSTM — treinamento concluído.')

## 9. Comparação de Resultados

In [ ]:
def mape(y_true, y_pred):
    """MAPE ignorando zeros para evitar divisão por zero."""
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def metricas(y_true, y_pred, nome):
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mp   = mape(y_true, y_pred)
    return {'Modelo': nome, 'RMSE': rmse, 'MAE': mae, 'MAPE (%)': mp}

# ── Tabela t+1 ──
resultados_t1 = pd.DataFrame([
    metricas(real_t1,      preds_arima_t1,  'ARIMA'),
    metricas(real_t1,      preds_enet_t1,   'Elastic Net'),   # ← linha nova
    metricas(real_t1,      preds_rf_t1,     'Random Forest'),
    metricas(real_lstm_t1, preds_lstm_t1,   'LSTM'),
]).set_index('Modelo').round(6)

# ── Tabela t+5 ──
resultados_t5 = pd.DataFrame([
    metricas(real_t5,      preds_arima_t5,  'ARIMA'),
    metricas(real_t5,      preds_enet_t5,   'Elastic Net'),   # ← linha nova
    metricas(real_t5,      preds_rf_t5,     'Random Forest'),
    metricas(real_lstm_t5, preds_lstm_t5,   'LSTM'),
]).set_index('Modelo').round(6)

print('═══════════════════════════════════════')
print('   RESULTADOS — Horizonte t+1 (1 dia)')
print('═══════════════════════════════════════')
print(resultados_t1.to_string())
print()
print('═══════════════════════════════════════')
print('   RESULTADOS — Horizonte t+5 (1 semana)')
print('═══════════════════════════════════════')
print(resultados_t5.to_string())

In [ ]:
# Visualização comparativa das métricas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metricas_nomes = ['RMSE', 'MAE', 'MAPE (%)']
modelos = ['ARIMA', 'Elastic Net', 'Random Forest', 'LSTM']
x = np.arange(len(modelos))  # agora 4 modelos, não 3
width = 0.35

for i, met in enumerate(metricas_nomes):
    vals_t1 = resultados_t1[met].values
    vals_t5 = resultados_t5[met].values
    axes[i].bar(x - width/2, vals_t1, width, label='t+1', color='#6A4C93')
    axes[i].bar(x + width/2, vals_t5, width, label='t+5', color='tomato')
    axes[i].set_title(met)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(modelos, rotation=15)
    axes[i].legend()

plt.suptitle('Comparação de Métricas — t+1 vs t+5', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/comparacao_metricas.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

## 9.5 Conformal Prediction (ACI)

In [ ]:
# ============================================================
# CÉLULA — Adaptive Conformal Inference (Gibbs & Candès, 2021)
# ============================================================

def aci_intervals(preds, real, alpha=0.10, gamma=0.02, burn_in=30, window=None,
                   alpha_inicial=None, scores_iniciais=None):
    """
    Adaptive Conformal Inference.
    alpha : taxa de erro alvo (0.10 -> cobertura nominal de 90%)
    gamma : taxa de adaptação do intervalo
    burn_in : nº de pontos iniciais usados só para acumular scores,
              sem contar cobertura (fase de aquecimento)
    window : se definido, mantém só os últimos N scores (janela deslizante);
             se None, usa todo o histórico acumulado
    alpha_inicial : permite continuar o ACI de um alpha_t já calibrado
                    (usado na validação fora da amostra, seção 9.7)
    scores_iniciais : permite continuar o ACI com um histórico de scores já
                       acumulado, em vez de começar do zero
    """
    n = len(preds)
    scores = list(scores_iniciais) if scores_iniciais is not None else []
    alpha_t = alpha_inicial if alpha_inicial is not None else alpha

    lower    = np.full(n, np.nan)
    upper    = np.full(n, np.nan)
    covered  = np.full(n, np.nan)
    alpha_hist = np.full(n, np.nan)

    for t in range(n):
        s_t = abs(real[t] - preds[t])

        if len(scores) >= 5:
            q_t = np.quantile(scores, min(max(1 - alpha_t, 0), 1))
        else:
            q_t = np.nan  # ainda em aquecimento

        lower[t] = preds[t] - q_t
        upper[t] = preds[t] + q_t
        alpha_hist[t] = alpha_t

        if t >= burn_in and not np.isnan(q_t):
            covered[t] = 1 if s_t <= q_t else 0
            err_t = 1 - covered[t]
            alpha_t = alpha_t + gamma * (alpha - err_t)
            alpha_t = min(max(alpha_t, 0.001), 0.999)

        scores.append(s_t)
        if window is not None and len(scores) > window:
            scores.pop(0)

    return lower, upper, covered, alpha_hist

# ── Parâmetros ──
ALPHA   = 0.10   # cobertura alvo de 90%
GAMMA   = 0.02
BURN_IN = 30

# ── Previsões e valores reais de cada modelo, já calculados nas seções 6–8 ──
modelos_preds = {
    'ARIMA':         (preds_arima_t1, real_t1,      preds_arima_t5, real_t5),
    'Elastic Net':   (preds_enet_t1,  real_t1,      preds_enet_t5,  real_t5),
    'Random Forest': (preds_rf_t1,    real_t1,      preds_rf_t5,    real_t5),
    'LSTM':          (preds_lstm_t1,  real_lstm_t1, preds_lstm_t5,  real_lstm_t5),
}

resultados_aci = {}

for nome, (p1, r1, p5, r5) in modelos_preds.items():
    lo1, hi1, cov1, a1 = aci_intervals(p1, r1, alpha=ALPHA, gamma=GAMMA, burn_in=BURN_IN)
    lo5, hi5, cov5, a5 = aci_intervals(p5, r5, alpha=ALPHA, gamma=GAMMA, burn_in=BURN_IN)

    resultados_aci[nome] = {
        't1': dict(lower=lo1, upper=hi1, cobertura=np.nanmean(cov1)*100,
                    largura=np.nanmean(hi1 - lo1), alpha_hist=a1),
        't5': dict(lower=lo5, upper=hi5, cobertura=np.nanmean(cov5)*100,
                    largura=np.nanmean(hi5 - lo5), alpha_hist=a5),
    }

print(f'Adaptive Conformal Inference — cobertura nominal: {(1-ALPHA)*100:.0f}%\n')
print(f'{"Modelo":15s} | {"t+1 cobertura":13s} | {"t+1 largura":12s} | {"t+5 cobertura":13s} | {"t+5 largura":12s}')
print('-' * 75)
for nome, res in resultados_aci.items():
    print(f'{nome:15s} | {res["t1"]["cobertura"]:11.1f}%  | {res["t1"]["largura"]:10.5f}  | '
          f'{res["t5"]["cobertura"]:11.1f}%  | {res["t5"]["largura"]:10.5f}')

In [ ]:
# ============================================================
# CÉLULA — Visualização dos intervalos ACI (t+1, últimos 120 dias)
# ============================================================

n_plot = 120
fig, axes = plt.subplots(4, 1, figsize=(16, 16))

for ax, (nome, (p1, r1, p5, r5)) in zip(axes, modelos_preds.items()):
    lo = resultados_aci[nome]['t1']['lower'][-n_plot:]
    hi = resultados_aci[nome]['t1']['upper'][-n_plot:]
    pred = p1[-n_plot:]
    real = r1[-n_plot:]
    xs = np.arange(len(real))

    ax.fill_between(xs, lo, hi, color='#6A4C93', alpha=0.2, label='Intervalo ACI (90%)')
    ax.plot(xs, real, color='#6A4C93', linewidth=1, label='Real')
    ax.plot(xs, pred, color='tomato', linewidth=1, linestyle='--', label='Previsto')
    ax.set_title(f'{nome} — Intervalo ACI t+1 (últimos {n_plot} dias)')
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/aci_intervalo_t1.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

In [ ]:
# ============================================================
# CÉLULA — Visualização dos intervalos ACI (t+5, últimos 120 dias)
# ============================================================

n_plot = 120
fig, axes = plt.subplots(4, 1, figsize=(16, 16))

for ax, (nome, (p1, r1, p5, r5)) in zip(axes, modelos_preds.items()):
    lo = resultados_aci[nome]['t5']['lower'][-n_plot:]
    hi = resultados_aci[nome]['t5']['upper'][-n_plot:]
    pred = p5[-n_plot:]
    real = r5[-n_plot:]
    xs = np.arange(len(real))

    ax.fill_between(xs, lo, hi, color='#6A4C93', alpha=0.2, label='Intervalo ACI (90%)')
    ax.plot(xs, real, color='#6A4C93', linewidth=1, label='Real')
    ax.plot(xs, pred, color='tomato', linewidth=1, linestyle='--', label='Previsto')
    ax.set_title(f'{nome} — Intervalo ACI t+5 (últimos {n_plot} dias)')
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/aci_intervalo_t5.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

In [ ]:
# ============================================================
# CÉLULA — Evolução do alpha_t ao longo do tempo (t+1)
# ============================================================

fig, ax = plt.subplots(figsize=(14, 4))
for nome in modelos_preds:
    ax.plot(resultados_aci[nome]['t1']['alpha_hist'], label=nome, linewidth=1)

ax.axhline(ALPHA, color='black', linestyle=':', linewidth=1, label=f'alpha nominal ({ALPHA})')
ax.set_title('Evolução de α_t — Adaptive Conformal Inference (t+1)')
ax.set_ylabel('α_t')
ax.set_xlabel('Dias de teste')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/aci_alpha_evolucao.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

## 9.6 Congelamento dos Modelos

In [ ]:
# ============================================================
# CÉLULA — Congelar modelos e estado do ACI para teste prospectivo
# ============================================================

import joblib

joblib.dump(enet_t1, '/kaggle/working/enet_t1.pkl')
joblib.dump(rf_t1,   '/kaggle/working/rf_t1.pkl')
joblib.dump(scaler_enet, '/kaggle/working/scaler_enet.pkl')

lstm_t1.save('/kaggle/working/lstm_t1.keras')
joblib.dump(scaler_lstm_features, '/kaggle/working/scaler_lstm_features.pkl')
joblib.dump(scaler_t1, '/kaggle/working/scaler_lstm_t1.pkl')

joblib.dump(arima_model, '/kaggle/working/arima_model.pkl')

estado_aci = {
    'ARIMA':         resultados_aci['ARIMA']['t1']['alpha_hist'][-1],
    'Elastic Net':   resultados_aci['Elastic Net']['t1']['alpha_hist'][-1],
    'Random Forest': resultados_aci['Random Forest']['t1']['alpha_hist'][-1],
    'LSTM':          resultados_aci['LSTM']['t1']['alpha_hist'][-1],
}
joblib.dump(estado_aci, '/kaggle/working/estado_aci.pkl')

df.tail(50).to_csv('/kaggle/working/historico_recente.csv')

print('Estado congelado com sucesso.')
print('Alpha_t inicial para o teste prospectivo:')
for nome, a in estado_aci.items():
    print(f'  {nome}: {a:.4f}')

## 9.7 Validação Fora da Amostra

In [ ]:
# ============================================================
# CÉLULA — 9.7 Validação Fora da Amostra (Out-of-Time)
# Período: mai/2025 até hoje — dados que os modelos NUNCA viram
# ============================================================

import yfinance as yf

# 1. Baixa dados desde um pouco antes de mai/2025
raw_oot = yf.download('EURUSD=X', start='2025-04-01', progress=False)

if isinstance(raw_oot.columns, pd.MultiIndex):
    raw_oot.columns = raw_oot.columns.get_level_values(0)

raw_oot = raw_oot[['Close']].rename(columns={'Close': 'EURUSD'})
raw_oot.index.name = 'Date'
raw_oot['retorno'] = np.log(raw_oot['EURUSD'] / raw_oot['EURUSD'].shift(1))

# 2. Features — mesmo pipeline da seção 5
raw_oot['ret_lag1']    = raw_oot['retorno'].shift(1)
raw_oot['ret_lag2']    = raw_oot['retorno'].shift(2)
raw_oot['ret_lag5']    = raw_oot['retorno'].shift(5)
raw_oot['media_mov5']  = raw_oot['EURUSD'].rolling(5).mean()
raw_oot['media_mov20'] = raw_oot['EURUSD'].rolling(20).mean()
raw_oot['vol_5d']      = raw_oot['retorno'].rolling(5).std()
raw_oot['vol_20d']     = raw_oot['retorno'].rolling(20).std()
raw_oot['evento']      = 0  # ajustar manualmente se houver eventos geopolíticos conhecidos no período

raw_oot['alvo_t1'] = raw_oot['retorno'].shift(-1)
raw_oot.dropna(inplace=True)

# 3. Recorta só o período realmente "não visto"
oot = raw_oot.loc['2025-05-01':]
print(f'Período OOT: {oot.index.min().date()} a {oot.index.max().date()} ({len(oot)} obs)')

X_oot = oot[FEATURES]

# 4. Previsões com os modelos CONGELADOS (sem retreinar)
X_oot_scaled = scaler_enet.transform(X_oot)
preds_enet_oot = enet_t1.predict(X_oot_scaled)
preds_rf_oot   = rf_t1.predict(X_oot)

# ARIMA — continua o rolling forecast de onde o 'history' original parou
history_oot = list(train['retorno']) + list(test['retorno'])
preds_arima_oot = []
for i in range(len(oot)):
    model_fit = arima_model.fit(history_oot)
    yhat = model_fit.predict(n_periods=1)[0]
    preds_arima_oot.append(yhat)
    history_oot.append(oot['retorno'].iloc[i])
preds_arima_oot = np.array(preds_arima_oot)

real_oot = oot['alvo_t1'].values

# 5. Métricas OOT
resultados_oot = pd.DataFrame([
    metricas(real_oot, preds_arima_oot, 'ARIMA'),
    metricas(real_oot, preds_enet_oot,  'Elastic Net'),
    metricas(real_oot, preds_rf_oot,    'Random Forest'),
]).set_index('Modelo').round(6)

print('\n═══════════════════════════════════════════')
print('   VALIDAÇÃO FORA DA AMOSTRA — mai/2025 a hoje')
print('═══════════════════════════════════════════')
print(resultados_oot.to_string())

# 6. ACI continuando do alpha_t final da seção 9.5
lo_enet, hi_enet, cov_enet, hist_enet = aci_intervals(
    preds_enet_oot, real_oot, alpha_inicial=estado_aci['Elastic Net'], burn_in=0)
lo_rf, hi_rf, cov_rf, hist_rf = aci_intervals(
    preds_rf_oot, real_oot, alpha_inicial=estado_aci['Random Forest'], burn_in=0)
lo_arima, hi_arima, cov_arima, hist_arima_alpha = aci_intervals(
    preds_arima_oot, real_oot, alpha_inicial=estado_aci['ARIMA'], burn_in=0)

alpha_final_enet  = hist_enet[-1]
alpha_final_rf    = hist_rf[-1]
alpha_final_arima = hist_arima_alpha[-1]

print(f'\nCobertura ACI (fora da amostra):')
print(f'  ARIMA:         {np.nanmean(cov_arima)*100:.1f}%')
print(f'  Elastic Net:   {np.nanmean(cov_enet)*100:.1f}%')
print(f'  Random Forest: {np.nanmean(cov_rf)*100:.1f}%')

# Atualiza o estado do ACI para a seção 9.8
estado_aci['ARIMA']         = alpha_final_arima
estado_aci['Elastic Net']   = alpha_final_enet
estado_aci['Random Forest'] = alpha_final_rf
joblib.dump(estado_aci, '/kaggle/working/estado_aci.pkl')
joblib.dump(history_oot, '/kaggle/working/history_arima_atualizado.pkl')
oot.tail(50).to_csv('/kaggle/working/historico_recente.csv')

# Salva os últimos scores da validação OOT — para a 9.8 já começar com
# intervalo válido no primeiro dia, sem precisar esperar acumular 5 erros novos
scores_iniciais = {
    'Elastic Net':   list(np.abs(real_oot - preds_enet_oot))[-30:],
    'Random Forest': list(np.abs(real_oot - preds_rf_oot))[-30:],
    'ARIMA':         list(np.abs(real_oot - preds_arima_oot))[-30:],
}
joblib.dump(scores_iniciais, '/kaggle/working/scores_iniciais.pkl')

print('\nEstado do ACI e scores iniciais atualizados para a seção 9.8.')

In [ ]:
# ============================================================
# CÉLULA — Visualização da Validação Fora da Amostra (mai/2025–hoje)
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

modelos_oot = [
    ('ARIMA',         preds_arima_oot, lo_arima, hi_arima),
    ('Elastic Net',   preds_enet_oot,  lo_enet,  hi_enet),
    ('Random Forest', preds_rf_oot,    lo_rf,    hi_rf),
]

xs = oot.index

for ax, (nome, pred, lo, hi) in zip(axes, modelos_oot):
    ax.fill_between(xs, lo, hi, color='#6A4C93', alpha=0.2, label='Intervalo ACI (90%)')
    ax.plot(xs, real_oot, color='#6A4C93', linewidth=1, label='Real')
    ax.plot(xs, pred, color='tomato', linewidth=1, linestyle='--', label='Previsto')
    ax.axvline(pd.Timestamp('2025-04-30'), color='gray', linestyle=':', linewidth=1,
               label='Fim do período de treino (abr/2025)')
    ax.set_title(f'{nome} — Validação Fora da Amostra (mai/2025 a hoje)')
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/validacao_oot.png', dpi=300, bbox_inches='tight')  # ajuste o nome por gráfico
plt.show()

## 9.8 Acompanhamento Prospectivo Diário

In [1]:
# ============================================================
# CÉLULA — 9.8 Acompanhamento Prospectivo Diário (autocontida)
# Rodar 1x por pregão, a partir de agora
# Lê o estado do dia anterior do Dataset "tcc-cambio-estado" e salva
# o estado atualizado em WORK_DIR (baixar e subir como Nova Versão
# do Dataset ao final, para virar o Input de amanhã)
# ============================================================

!pip install pmdarima --quiet

import os
import shutil
import joblib
import numpy as np
import pandas as pd
import yfinance as yf
import tensorflow as tf
from datetime import datetime

WORK_DIR  = '/kaggle/working'
INPUT_DIR = '/kaggle/input/datasets/kelwinpaschoal/tcc-cambio-estado'   # ← única linha alterada
LOG_PATH  = f'{WORK_DIR}/log_prospectivo.csv'
ALPHA_NOMINAL = 0.10
GAMMA = 0.02

FEATURES = ['ret_lag1', 'ret_lag2', 'ret_lag5',
            'media_mov5', 'media_mov20',
            'vol_5d', 'vol_20d', 'evento']

# ── 1. Carrega modelos e estado salvos (do INPUT_DIR — somente leitura) ──
enet_t1     = joblib.load(f'{INPUT_DIR}/enet_t1.pkl')
rf_t1       = joblib.load(f'{INPUT_DIR}/rf_t1.pkl')
scaler_enet = joblib.load(f'{INPUT_DIR}/scaler_enet.pkl')
arima_model = joblib.load(f'{INPUT_DIR}/arima_model.pkl')
estado_aci  = joblib.load(f'{INPUT_DIR}/estado_aci.pkl')
history_arima = joblib.load(f'{INPUT_DIR}/history_arima_atualizado.pkl')
scores_iniciais = joblib.load(f'{INPUT_DIR}/scores_iniciais.pkl')

hist = pd.read_csv(f'{INPUT_DIR}/historico_recente.csv', index_col=0, parse_dates=True)
hist.index = pd.to_datetime(hist.index)

LOG_PATH_INPUT = f'{INPUT_DIR}/log_prospectivo.csv'
if os.path.exists(LOG_PATH_INPUT) and not os.path.exists(LOG_PATH):
    shutil.copy(LOG_PATH_INPUT, LOG_PATH)

# ── 2. Baixa dados atualizados e recalcula features ─────────
novos = yf.download('EURUSD=X', start=(hist.index.min() - pd.Timedelta(days=5)).date(),
                     progress=False)

if isinstance(novos.columns, pd.MultiIndex):
    novos.columns = novos.columns.get_level_values(0)

novos = novos[['Close']].rename(columns={'Close': 'EURUSD'})
novos.index.name = 'Date'
novos['retorno'] = np.log(novos['EURUSD'] / novos['EURUSD'].shift(1))

df_atual = novos.combine_first(hist[['EURUSD', 'retorno']]).sort_index()
df_atual = df_atual[~df_atual.index.duplicated(keep='last')]

df_atual['ret_lag1']    = df_atual['retorno'].shift(1)
df_atual['ret_lag2']    = df_atual['retorno'].shift(2)
df_atual['ret_lag5']    = df_atual['retorno'].shift(5)
df_atual['media_mov5']  = df_atual['EURUSD'].rolling(5).mean()
df_atual['media_mov20'] = df_atual['EURUSD'].rolling(20).mean()
df_atual['vol_5d']      = df_atual['retorno'].rolling(5).std()
df_atual['vol_20d']     = df_atual['retorno'].rolling(20).std()
df_atual['evento']      = 0  # ajustar manualmente se hoje for dia de evento geopolítico
df_atual.dropna(inplace=True)

hoje = df_atual.index[-1]
linha_hoje = df_atual.iloc[[-1]]
retorno_realizado_hoje = linha_hoje['retorno'].values[0]

print(f'Data processada: {hoje.date()}')

# ── 3. Confere a previsão feita no pregão anterior (se houver) ──
if os.path.exists(LOG_PATH):
    log = pd.read_csv(LOG_PATH, parse_dates=['data_previsao_para'])
    pendente = log[log['real'].isna()]

    if not pendente.empty:
        idx_pendente = pendente.index[-1]
        data_alvo = log.loc[idx_pendente, 'data_previsao_para']

        if pd.Timestamp(data_alvo).normalize() == hoje.normalize():
            log.loc[idx_pendente, 'real'] = retorno_realizado_hoje

            for modelo, col_pred in [('enet', 'pred_enet'), ('rf', 'pred_rf'), ('arima', 'pred_arima')]:
                pred = log.loc[idx_pendente, col_pred]
                lower_col, upper_col = f'{col_pred}_lower', f'{col_pred}_upper'
                lo, hi = log.loc[idx_pendente, lower_col], log.loc[idx_pendente, upper_col]
                coberto = 1 if (lo <= retorno_realizado_hoje <= hi) else 0
                log.loc[idx_pendente, f'{col_pred}_coberto'] = coberto

                nome_modelo = {'enet': 'Elastic Net', 'rf': 'Random Forest', 'arima': 'ARIMA'}[modelo]
                estado_aci[nome_modelo] = min(max(
                    estado_aci[nome_modelo] + GAMMA * (ALPHA_NOMINAL - (1 - coberto)), 0.001), 0.999)

            log.to_csv(LOG_PATH, index=False)
            print('Previsão de ontem conferida e log atualizado.')

            cobertura_acumulada = {
                'Elastic Net':   log['pred_enet_coberto'].mean() * 100,
                'Random Forest': log['pred_rf_coberto'].mean() * 100,
                'ARIMA':         log['pred_arima_coberto'].mean() * 100,
            }
            print('\nCobertura acumulada até agora:')
            for nome, cov in cobertura_acumulada.items():
                if not np.isnan(cov):
                    print(f'  {nome}: {cov:.1f}%  (nominal: {(1-ALPHA_NOMINAL)*100:.0f}%)')
else:
    log = pd.DataFrame(columns=[
        'data_referencia', 'data_previsao_para',
        'pred_enet', 'pred_enet_lower', 'pred_enet_upper', 'pred_enet_coberto',
        'pred_rf', 'pred_rf_lower', 'pred_rf_upper', 'pred_rf_coberto',
        'pred_arima', 'pred_arima_lower', 'pred_arima_upper', 'pred_arima_coberto',
        'real'
    ])
    print('Nenhum log anterior encontrado — iniciando novo acompanhamento.')

# ── 4. Gera a previsão para o PRÓXIMO pregão ─────────────────
X_hoje = linha_hoje[FEATURES]
X_hoje_scaled = scaler_enet.transform(X_hoje)

pred_enet = enet_t1.predict(X_hoje_scaled)[0]
pred_rf   = rf_t1.predict(X_hoje)[0]

history_arima.append(retorno_realizado_hoje)
model_fit = arima_model.fit(history_arima)
pred_arima = model_fit.predict(n_periods=1)[0]
joblib.dump(history_arima, f'{WORK_DIR}/history_arima_atualizado.pkl')

# ── 5. Calcula os intervalos ACI com o alpha_t atual ─────────
def quantil_intervalo(alpha_t, scores_recentes):
    if len(scores_recentes) < 5:
        return np.nan
    return np.quantile(scores_recentes, min(max(1 - alpha_t, 0), 1))

scores_enet  = scores_iniciais['Elastic Net']   + ((log['real'] - log['pred_enet']).abs().dropna().tolist()  if len(log) else [])
scores_rf    = scores_iniciais['Random Forest'] + ((log['real'] - log['pred_rf']).abs().dropna().tolist()    if len(log) else [])
scores_arima = scores_iniciais['ARIMA']         + ((log['real'] - log['pred_arima']).abs().dropna().tolist() if len(log) else [])

q_enet  = quantil_intervalo(estado_aci['Elastic Net'], scores_enet)
q_rf    = quantil_intervalo(estado_aci['Random Forest'], scores_rf)
q_arima = quantil_intervalo(estado_aci['ARIMA'], scores_arima)

proximo_pregao = pd.bdate_range(hoje, periods=2)[-1]

nova_linha = pd.DataFrame([{
    'data_referencia': hoje.date(),
    'data_previsao_para': proximo_pregao,
    'pred_enet': pred_enet, 'pred_enet_lower': pred_enet - q_enet, 'pred_enet_upper': pred_enet + q_enet, 'pred_enet_coberto': np.nan,
    'pred_rf': pred_rf, 'pred_rf_lower': pred_rf - q_rf, 'pred_rf_upper': pred_rf + q_rf, 'pred_rf_coberto': np.nan,
    'pred_arima': pred_arima, 'pred_arima_lower': pred_arima - q_arima, 'pred_arima_upper': pred_arima + q_arima, 'pred_arima_coberto': np.nan,
    'real': np.nan,
}])

log = pd.concat([log, nova_linha], ignore_index=True)
log.to_csv(LOG_PATH, index=False)

joblib.dump(estado_aci, f'{WORK_DIR}/estado_aci.pkl')
df_atual.tail(50).to_csv(f'{WORK_DIR}/historico_recente.csv')
joblib.dump(scores_iniciais, f'{WORK_DIR}/scores_iniciais.pkl')

print(f'\nPrevisão gerada para {proximo_pregao.date()}:')
print(f'  Elastic Net:   {pred_enet:.6f}  [{pred_enet - q_enet:.6f}, {pred_enet + q_enet:.6f}]')
print(f'  Random Forest: {pred_rf:.6f}  [{pred_rf - q_rf:.6f}, {pred_rf + q_rf:.6f}]')
print(f'  ARIMA:         {pred_arima:.6f}  [{pred_arima - q_arima:.6f}, {pred_arima + q_arima:.6f}]')

print('\n' + '='*60)
print('LEMBRETE: baixe estado_aci.pkl, history_arima_atualizado.pkl,')
print('scores_iniciais.pkl, historico_recente.csv e log_prospectivo.csv')
print('do painel de output, e suba como Nova Versão do dataset')
print('"tcc-cambio-estado" antes de fechar a sessão.')
print('='*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.1 MB/s eta 0:00:00 0:00:01


/tmp/ipykernel_58/3476836894.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  novos = yf.download('EURUSD=X', start=(hist.index.min() - pd.Timedelta(days=5)).date(),


Data processada: 2026-09-15
Previsão de ontem conferida e log atualizado.

Cobertura acumulada até agora:
  Elastic Net: 75.0%  (nominal: 90%)
  Random Forest: 75.0%  (nominal: 90%)
  ARIMA: 75.0%  (nominal: 90%)

Previsão gerada para 2026-09-16:
  Elastic Net:   -0.000074  [-0.003830, 0.003682]
  Random Forest: -0.000181  [-0.004002, 0.003639]
  ARIMA:         0.000099  [-0.003567, 0.003764]

LEMBRETE: baixe estado_aci.pkl, history_arima_atualizado.pkl,
scores_iniciais.pkl, historico_recente.csv e log_prospectivo.csv
do painel de output, e suba como Nova Versão do dataset
"tcc-cambio-estado" antes de fechar a sessão.


## 10. Conclusões

Esta célula deve ser preenchida **após** a execução do notebook com os resultados reais obtidos.

Sugestão de estrutura para o texto do TCC:

**Comparação geral:** qual modelo apresentou menor RMSE/MAE/MAPE em t+1 e t+5?

**Impacto dos eventos geopolíticos:** o gráfico de importância de variáveis do Random Forest mostra o peso da variável `evento`? Como a volatilidade nos períodos de evento se compara aos períodos normais?

**Limitações:** os modelos preveem *direção* ou *magnitude*? Para câmbio, acertar a direção pode ser mais relevante que minimizar o erro absoluto.

**Trabalhos futuros:** incorporar dados macroeconômicos (inflação, juros) como preditores adicionais, a exemplo de Filippou et al. (2022).